<a href="https://colab.research.google.com/github/RuchiKalkandha/Vision-Transformers/blob/main/ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision


In [ ]:
import torchvision.transforms as transforms

In [ ]:
#transformation for PIL to tensor format
transformation_operation = transforms.Compose([transforms.ToTensor()])

In [ ]:
train_dataset = torchvision.datasets.MNIST(root = './data', train = True, download=True, transform = transformation_operation)
val_dataset = torchvision.datasets.MNIST(root = './data', train = False , download = True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 511kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.67MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.96MB/s]


In [ ]:
import torch.utils.data as dataLoader

In [ ]:
#Defining dataset batches
train_loader =  dataLoader.DataLoader(dataset = train_dataset, batch_size = 64, shuffle = True)
val_loader = dataLoader.DataLoader(dataset = val_dataset, batch_size = 64, shuffle= True)

In [ ]:
num_classes = 10
batch_size = 64
num_channels = 1
image_size = 28
patch_size = 7
num_patches = (image_size // patch_size) ** 2
embedding_dim = 64
attention_heads = 4
transformer_blocks = 4
mlp_hidden_nodes = 128
epochs = 5
learning_rate = 0.001


In [ ]:
import torch.nn as nn

In [ ]:
class PatchEmbedding(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embed = nn.Conv2d(num_channels, embedding_dim, kernel_size= patch_size, stride = patch_size )

  def forward(self, x):
      x = self.patch_embed(x)
      x = x.flatten(2).transpose(1,2)
      return x

In [ ]:
class TransformerEncoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(embedding_dim)
    self.layer_norm2 = nn.LayerNorm(embedding_dim)
    self.multihead_attention = nn.MultiheadAttention(embedding_dim, attention_heads, batch_first= True)
    self.mlp = nn.Sequential(
        nn.Linear(embedding_dim, mlp_hidden_nodes),
        nn.GELU(),
        nn.Linear(mlp_hidden_nodes, embedding_dim)
    )
  def forward(self, x):
    residual1 = x
    x = self.layer_norm1(x)
    x = self.multihead_attention(x, x, x)[0]
    x = residual1 + x
    residual2 = x
    x = self.layer_norm2(x)
    x = self.mlp(x)
    x = residual2 + x
    return x

In [ ]:
class MLP_head(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(embedding_dim)
    self.mlp_head = nn.Linear(embedding_dim, num_classes)

  def forward(self, x):
    x = self.layer_norm1(x)
    x = self.mlp_head(x)
    return x

In [ ]:
class VisionTransformer(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embedding = PatchEmbedding()
    self.class_token = nn.Parameter(torch.randn(1, 1, embedding_dim))
    self.position_embedding = nn.Parameter(torch.randn(1, num_patches + 1, embedding_dim))
    self.transformer_blocks = nn.Sequential(*[TransformerEncoder() for _ in range(transformer_blocks)])
    self.mlp_head = MLP_head()

  def forward(self, x):
    x = self.patch_embedding(x)
    class_token = self.class_token.expand(x.shape[0], -1, -1)
    x = torch.cat(( class_token, x), dim = 1)
    x = x + self.position_embedding
    x = self.transformer_blocks(x)
    x = x[:,0]
    x = self.mlp_head(x)
    return x

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VisionTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
criterion = nn.CrossEntropyLoss()


In [ ]:
#training
for epoch in range(epochs):
  model.train()
  total_loss = 0
  correct_epoch = 0 # Initialize correct_epoch
  total_epoch = 0
  print(f"\nEpoch {epoch+1}")

  for batch_idx, (images, labels) in enumerate(train_loader):
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    preds = outputs.argmax(dim = 1)
    correct = (preds == labels).sum().item()
    accuracy = 100.0 * correct / labels.size(0)
    correct_epoch += correct
    total_epoch += labels.size(0)
    if batch_idx % 100 == 0:
      print(f" Batch {batch_idx+1:3d}: Loss = {loss.item(): .4f}, Accuracy = {accuracy:.2f}%")
  epoch_acc = 100.0 * correct_epoch / total_epoch
  print(f"Epoch {epoch+1} Summary : Total Loss = {total_loss: .4f}, Accuracy = {epoch_acc: .2f}% ")


Epoch 1
 Batch   1: Loss =  2.4013, Accuracy = 9.38%
 Batch 101: Loss =  0.5243, Accuracy = 85.94%
 Batch 201: Loss =  0.2577, Accuracy = 90.62%
 Batch 301: Loss =  0.2767, Accuracy = 89.06%
 Batch 401: Loss =  0.3298, Accuracy = 89.06%
 Batch 501: Loss =  0.2655, Accuracy = 90.62%
 Batch 601: Loss =  0.1098, Accuracy = 98.44%
 Batch 701: Loss =  0.1333, Accuracy = 96.88%
 Batch 801: Loss =  0.0866, Accuracy = 96.88%
 Batch 901: Loss =  0.1969, Accuracy = 96.88%
Epoch 1 Summary : Total Loss =  328.3635, Accuracy =  89.11% 

Epoch 2
 Batch   1: Loss =  0.1325, Accuracy = 93.75%
 Batch 101: Loss =  0.1008, Accuracy = 95.31%
 Batch 201: Loss =  0.0489, Accuracy = 98.44%
 Batch 301: Loss =  0.1419, Accuracy = 96.88%
 Batch 401: Loss =  0.1188, Accuracy = 96.88%
 Batch 501: Loss =  0.0797, Accuracy = 96.88%
 Batch 601: Loss =  0.0848, Accuracy = 96.88%
 Batch 701: Loss =  0.0261, Accuracy = 100.00%
 Batch 801: Loss =  0.0909, Accuracy = 98.44%
 Batch 901: Loss =  0.0321, Accuracy = 100.00%